# stray_model_v5 — เทรนแบบ K-Fold Cross-Validation

**เปลี่ยนจาก v4.3 อย่างไร และทำไม**

| หัวข้อ | v4.3 (เดิม) | v5 (ใหม่) | เหตุผล |
|---|---|---|---|
| การแบ่งข้อมูล | Holdout ครั้งเดียว 70/15/15 | **กัน Test 15% ออกก่อน แล้วทำ Stratified 5-Fold บนอีก 85%** | Holdout ให้ตัวเลขชุดเดียว ไม่รู้ว่าฟลุคหรือเปล่า K-Fold ให้ค่าเฉลี่ย ± ส่วนเบี่ยงเบน |
| ตัวเลขที่รายงาน | Accuracy ค่าเดียว | **mean ± std จาก 5 fold + ค่าบน Test set** | บอกได้ว่าผลเสถียรแค่ไหน |
| Oversample / Class weight | คำนวณครั้งเดียวก่อนแบ่ง | **คำนวณใหม่ในทุก fold จากเฉพาะส่วน train ของ fold นั้น** | ถ้าคำนวณก่อนแบ่งจะเกิด data leakage |
| โมเดล | สร้างครั้งเดียว | **สร้างใหม่ทุก fold** | ไม่งั้น fold หลังจะได้เปรียบเพราะเห็นข้อมูลของ fold ก่อนมาแล้ว |
| รันไม่จบ | ต้องเริ่มใหม่หมด | **Resume ได้ — รันต่อจาก fold ที่ค้างไว้** | Colab ฟรีหลุดบ่อย 5 fold ใช้เวลาหลายชั่วโมง |

**สิ่งที่ยังเหมือนเดิมทุกอย่าง** — EfficientNetB0, เทรน 3 phase, ล็อก BatchNorm, Focal Loss, MixUp เฉพาะ Phase 1, letterbox 300x300

---

**เวลาที่ใช้:** ประมาณ 1.5 ชั่วโมงต่อ fold บน GPU T4 → 5 fold รวม **7-8 ชั่วโมง** รันรวดเดียวไม่จบแน่นอน
ระบบ resume ทำไว้ให้แล้ว ให้รันวันละ 1-2 fold แล้วปิดไปได้เลย ครั้งหน้ามารันต่อ cell เดิมได้

**วิธีใช้:** รันทีละ cell ตามลำดับ ห้ามข้าม


## Cell 1 — เชื่อม Drive + เช็ค GPU

**ควรได้:** `Mounted at /content/drive` และ `GPU: /device:GPU:0`
ถ้าขึ้น `ไม่พบ` ให้ไป Runtime -> Change runtime type -> เลือก GPU (T4) ก่อน

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpu = tf.config.list_physical_devices('GPU')
print("GPU:", gpu[0].name if gpu else "ไม่พบ (ไปเปลี่ยน Runtime เป็น GPU ก่อน)")


## Cell 2 — ฟอนต์ไทยสำหรับกราฟ

**ควรได้:** `ตั้งค่าฟอนต์ไทยสำเร็จ` — ต้องรันก่อนกราฟทุกอัน ไม่งั้นตัวอักษรไทยจะเป็นกล่องสี่เหลี่ยม

In [ ]:
!apt-get -qq install -y fonts-thai-tlwg > /dev/null 2>&1

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import glob

for font_file in glob.glob('/usr/share/fonts/**/*.ttf', recursive=True):
    if any(n in font_file.lower() for n in ['loma','garuda','norasi','kinnari','sawasdee','tlwg']):
        fm.fontManager.addfont(font_file)

plt.rcParams['font.family'] = 'Loma'
plt.rcParams['axes.unicode_minus'] = False
print("ตั้งค่าฟอนต์ไทยสำเร็จ")


## Cell 3 — ค่าคงที่ทั้งหมด (Config)

**จุดที่เปลี่ยนจาก v4.3:** เพิ่ม `N_FOLDS` กับ `TEST_RATIO`

`TEST_RATIO = 0.15` คือส่วนที่**กันออกไปก่อนทำ K-Fold และห้ามแตะจนกว่าจะจบ**
ที่เหลืออีก 85% เรียกว่า dev set จะถูกแบ่งเป็น 5 fold

In [ ]:
import os

RUN_NAME   = 'stray_model_v5_kfold'
DATA_DIR   = '/content/stray_dataset'
SAVE_PATH  = f'/content/drive/MyDrive/{RUN_NAME}'
os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

IMG_SIZE    = 300
BATCH_SIZE  = 32
SEED        = 42

N_FOLDS     = 5      # จำนวน fold — 5 เป็นค่ามาตรฐาน ถ้าเวลาน้อยจริงๆ ใช้ 3 ได้แต่ค่าจะแกว่งกว่า
TEST_RATIO  = 0.15   # กันไว้วัดผลสุดท้าย ห้ามให้โมเดลเห็นระหว่าง CV เด็ดขาด

OVERSAMPLE_CAP_RATIO = 0.5

# จำนวน epoch ต่อ 1 fold (ค่าสะสม ไม่ใช่จำนวนรอบเพิ่ม)
EPOCHS_PHASE1 = 10   # หัวโมเดลอย่างเดียว
EPOCHS_PHASE2 = 25   # สะสมถึง 25 -> Phase 2 ได้ 15 รอบ
EPOCHS_PHASE3_EXTRA = 20   # ต่อจาก Phase 2 อีก 20 รอบ

print("RUN_NAME :", RUN_NAME)
print("SAVE_PATH:", SAVE_PATH)
print(f"แบ่ง: Test {TEST_RATIO:.0%} | dev set ที่เหลือแบ่งเป็น {N_FOLDS} fold")


## Cell 4 — เตรียม dataset 31 สายพันธุ์ (เหมือน v4.3 ทุกประการ)

**ควรได้:** บรรทัดสุดท้ายต้องนับได้ **31 สายพันธุ์** เสมอ
ถ้ามี `stray_dataset_v4_31breeds.zip` ใน Drive อยู่แล้วจะใช้ทางลัด ไม่ต้องโหลด Oxford ใหม่

In [ ]:
import os, re, shutil, tarfile, zipfile, urllib.request
from pathlib import Path

os.makedirs(DATA_DIR, exist_ok=True)

# ไฟล์ dataset รวมทั้ง 31 สายพันธุ์ (Oxford 18 + เทรนเพิ่มเอง 13) เก็บไว้ที่เดียวบน Drive
# มีแล้ว -> โหลดทางลัด | ยังไม่มี -> สร้างครั้งแรก แล้วเก็บไว้ให้ครั้งหน้าใช้ทางลัดได้เลย
COMBINED_ZIP = '/content/drive/MyDrive/stray_dataset_v4_31breeds.zip'

if os.path.exists(COMBINED_ZIP):
    # ---- ทางลัด: มี dataset รวมอยู่ใน Drive แล้ว ก็อปปี้มาแตกที่ disk ของ Colab ----
    print("เจอไฟล์ dataset รวมใน Drive แล้ว กำลังคัดลอกมาที่ disk ของ Colab (ไฟล์เดียว เร็วกว่าโหลดรูปทีละไฟล์มาก)...")
    local_zip = '/content/stray_dataset_v4_31breeds.zip'
    shutil.copy(COMBINED_ZIP, local_zip)
    with zipfile.ZipFile(local_zip, 'r') as z:
        z.extractall(DATA_DIR)
    print("แตกไฟล์จาก Drive สำเร็จ — ข้ามการดาวน์โหลด Oxford-IIIT และการอัปโหลด zip สายพันธุ์เพิ่มเองไปเลย")

else:
    # ---- ทางยาว (ทำครั้งแรกครั้งเดียว): ดาวน์โหลด Oxford + รวมกับสายพันธุ์ที่เทรนเพิ่มเอง ----
    print("ยังไม่มีไฟล์ dataset รวมใน Drive จะดาวน์โหลด/รวมข้อมูลใหม่ (ทำครั้งเดียว ครั้งหน้าจะเร็วขึ้นมาก)")

    OXFORD_KEEP = {
        # หมา (11)
        'beagle', 'pug', 'english_cocker_spaniel', 'american_bulldog',
        'american_pit_bull_terrier', 'pomeranian', 'chihuahua', 'shiba_inu',
        'yorkshire_terrier', 'havanese', 'japanese_chin',
        # แมว (7)
        'british_shorthair', 'persian', 'birman', 'ragdoll', 'siamese',
        'maine_coon', 'bengal',
    }

    oxford_tar = '/content/oxford_images.tar.gz'
    if not os.path.exists(oxford_tar):
        print("กำลังดาวน์โหลด Oxford-IIIT Pet Dataset (~800MB, รอสักครู่)...")
        urllib.request.urlretrieve(
            'https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz', oxford_tar
        )
        print("ดาวน์โหลดเสร็จ")

    pattern = re.compile(r'^images/(.+)_\d+\.(jpg|jpeg|png)$', re.IGNORECASE)
    คัดลอกแล้ว = 0
    with tarfile.open(oxford_tar, 'r:gz') as tar:
        for member in tar.getmembers():
            m = pattern.match(member.name)
            if not m:
                continue
            breed = m.group(1).lower()
            if breed not in OXFORD_KEEP:
                continue
            breed_dir = Path(DATA_DIR) / breed
            breed_dir.mkdir(exist_ok=True)
            fileobj = tar.extractfile(member)
            if fileobj is None:
                continue
            out_path = breed_dir / Path(member.name).name
            with open(out_path, 'wb') as f:
                shutil.copyfileobj(fileobj, f)
            คัดลอกแล้ว += 1
    print(f"แตกไฟล์ Oxford-IIIT (เฉพาะ {len(OXFORD_KEEP)} สายพันธุ์ที่เก็บ) แล้ว {คัดลอกแล้ว} รูป")

    # สายพันธุ์ที่เทรนเพิ่มเอง — ต้องอัปโหลด zip นี้ไว้ที่ Drive ก่อน (ใช้แค่รอบแรกที่สร้าง dataset รวม)
    CUSTOM_ZIP = '/content/drive/MyDrive/stray_custom_breeds.zip'
    if os.path.exists(CUSTOM_ZIP):
        with zipfile.ZipFile(CUSTOM_ZIP, 'r') as z:
            z.extractall(DATA_DIR)
        print("แตกไฟล์สายพันธุ์ที่เทรนเพิ่มเองสำเร็จ")
    else:
        print(f"\u26a0\ufe0f ไม่พบไฟล์ {CUSTOM_ZIP} — อัปโหลด zip ไปที่ Drive ก่อน แล้วรัน cell นี้ใหม่")

    # ---- รวมทุกอย่างที่เพิ่งได้มาเป็น zip เดียว เก็บไว้ใน Drive ให้ครั้งหน้าโหลดทางลัดได้เลย ----
    print("\nกำลังรวม dataset ทั้งหมดเป็น zip ไฟล์เดียว เก็บไว้ใน Drive (ทำครั้งเดียว รอสักครู่)...")
    local_zip_noext = '/content/stray_dataset_v4_31breeds'
    shutil.make_archive(local_zip_noext, 'zip', DATA_DIR)
    shutil.copy(local_zip_noext + '.zip', COMBINED_ZIP)
    print(f"บันทึก dataset รวมไว้ที่ {COMBINED_ZIP} แล้ว — เทรนครั้งหน้าจะโหลดจากไฟล์นี้ได้เลย ไม่ต้องดาวน์โหลด Oxford ซ้ำ")

print("\nสายพันธุ์ทั้งหมดใน DATA_DIR ตอนนี้:", len(os.listdir(DATA_DIR)), "สายพันธุ์")
print(sorted(os.listdir(DATA_DIR)))


## Cell 5 — สแกนไฟล์ทั้งหมด สร้างตาราง (filepath, label)

**ควรได้:** จำนวนรูปรวม 6,865 รูป และ 31 คลาส

In [ ]:
import pandas as pd
from pathlib import Path

CLASS_NAMES = sorted(os.listdir(DATA_DIR))
records = []
for label in CLASS_NAMES:
    folder = Path(DATA_DIR) / label
    if not folder.is_dir():
        continue
    for f in folder.glob('*'):
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'):
            records.append({'filepath': str(f), 'label': label})

df = pd.DataFrame(records)
print("จำนวนรูปทั้งหมด:", len(df))
print("จำนวนสายพันธุ์:", df['label'].nunique())
df.head()


## Cell 6 — ดูการกระจายตัวของข้อมูล (ก่อนปรับสมดุล)

**ควรได้:** กราฟแท่ง เห็นชัดว่าไม่สมดุล — มากสุดประมาณ 345 รูป น้อยสุดประมาณ 107 รูป

In [ ]:
import matplotlib.pyplot as plt

counts = df['label'].value_counts()

plt.figure(figsize=(14, 6))
counts.plot(kind='bar')
plt.title('จำนวนรูปต่อสายพันธุ์ (ก่อน oversample)')
plt.ylabel('จำนวนรูป')
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}/dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"บันทึกรูปกราฟไว้ที่ {SAVE_PATH}/dataset_distribution.png")

print("สายพันธุ์ที่มีรูปน้อยที่สุด 5 อันดับ:")
print(counts.tail(5))


## Cell 7 — กัน Test set ออกก่อน แล้วค่อยแบ่ง K-Fold ⭐ จุดที่ต่างจาก v4.3

**ลำดับสำคัญมาก ห้ามสลับ**

```
ข้อมูลทั้งหมด 6,865 รูป
   |
   +-- Test set 15% (~1,030 รูป)  <- ล็อกตู้เซฟ ห้ามแตะจนจบงาน
   |
   +-- dev set 85% (~5,835 รูป)   <- เอาไปแบ่ง 5 fold
          fold 1: train 80% / val 20%
          fold 2: train 80% / val 20%   ... ไปจนครบ 5 fold
```

**ทำไมต้องกัน Test ออกก่อน** ถ้าเอาข้อมูลทั้งหมดไปทำ K-Fold เลย จะไม่เหลือชุดที่โมเดล
"ไม่เคยเห็นเลยแม้แต่ตอนเลือกโมเดล" ค่าที่ได้จะดูดีเกินจริง เพราะเราใช้ค่า CV ไปตัดสินใจ
หลายอย่าง (เลือก epoch ที่ดีที่สุด เลือก fold ที่ดีที่สุด) ซึ่งนับเป็นการ "แอบดู" ข้อมูลไปแล้ว

**ควรได้:** Test ประมาณ 1,030 รูป | dev ประมาณ 5,835 รูป | รวมเท่ากับ Cell 5

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
import numpy as np

# stratify=df['label'] -> ทุกสายพันธุ์ถูกแบ่งตามสัดส่วนเดียวกัน
# สายพันธุ์ที่มี 107 รูปจึงได้เข้า test ประมาณ 16 รูปแน่นอน ไม่ใช่ 0 รูป
dev_df, test_df = train_test_split(
    df, test_size=TEST_RATIO, stratify=df['label'], random_state=SEED
)
dev_df  = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"dev set (เอาไปทำ {N_FOLDS}-Fold): {len(dev_df):,} รูป")
print(f"Test set (ล็อกไว้)            : {len(test_df):,} รูป")
print(f"รวม                           : {len(dev_df)+len(test_df):,} รูป  (ต้องเท่ากับ Cell 5)")

# เก็บรายชื่อไฟล์ test ลง Drive — ถ้ามารันใหม่วันหลังจะได้ test set ชุดเดิมเป๊ะ
test_df.to_csv(f'{SAVE_PATH}/test_set_files.csv', index=False, encoding='utf-8-sig')
print(f"บันทึกรายชื่อไฟล์ Test set ไว้ที่ {SAVE_PATH}/test_set_files.csv แล้ว")


## Cell 8 — สร้าง 5 Fold แบบ Stratified + ดูจำนวนต่อ fold

**ควรได้:** ตาราง 5 แถว จำนวน train/val ใกล้เคียงกันทุก fold และคอลัมน์สุดท้าย
(จำนวนรูปของสายพันธุ์ที่หายากที่สุดใน val) ต้องมากกว่า 0 ทุก fold

**จุดสำคัญ:** ทุกรูปใน dev set จะได้เป็น validation **หนึ่งครั้งพอดี** ไม่ซ้ำไม่ขาด
แปลว่าสายพันธุ์ Suphalak ที่มี 91 รูปใน dev set จะถูกใช้ประเมินครบทั้ง 91 รูป
ต่างจาก holdout เดิมที่ประเมินจากแค่ 16 รูป

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

y_dev = dev_df['label'].values
folds = list(skf.split(np.zeros(len(dev_df)), y_dev))

สายพันธุ์หายาก = dev_df['label'].value_counts().idxmin()
แถว = []
for i, (tr_idx, va_idx) in enumerate(folds):
    va_labels = dev_df.iloc[va_idx]['label']
    แถว.append({
        'fold': i + 1,
        'train': len(tr_idx),
        'val': len(va_idx),
        f'val ของ {สายพันธุ์หายาก}': int((va_labels == สายพันธุ์หายาก).sum())
    })

import pandas as pd
สรุปfold = pd.DataFrame(แถว)
display(สรุปfold)
print(f"\nรวม val ของทุก fold = {สรุปfold['val'].sum():,} รูป (ต้องเท่ากับ dev set = {len(dev_df):,})")


## Cell 9 — tf.data pipeline (letterbox 300x300) — เหมือน v4.3

**จุดสำคัญ:** ใช้ `resize_with_pad` (letterbox) ให้ตรงกับที่ `main.py` ทำตอนใช้งานจริง

In [ ]:
# map ชื่อคลาส -> ดัชนี (ย้ายมาจาก cell class weight ของ v4.3 เพราะ v5 ใช้ตั้งแต่ตรงนี้)
label_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}

AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    # decode_image รองรับ jpg/png/webp — เดิมใช้ decode_jpeg ซึ่งพังถ้าเจอ .png/.webp
    # ที่ปนมาในสายพันธุ์ที่เราเก็บรูปเพิ่มเอง (Cell 3 กวาดนามสกุลพวกนี้เข้ามาด้วย)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])   # decode_image ไม่คืน shape ต้องบอกเองก่อน resize
    img = tf.image.resize_with_pad(img, IMG_SIZE, IMG_SIZE)  # letterbox — ต้องตรงกับ main.py
    img = tf.cast(img, tf.float32)
    return img, label

def build_dataset(dataframe, shuffle=False, batch_size=BATCH_SIZE):
    labels_idx = dataframe['label'].map(label_to_idx).values
    labels_onehot = tf.keras.utils.to_categorical(labels_idx, num_classes=len(CLASS_NAMES)).astype('float32')
    ds = tf.data.Dataset.from_tensor_slices((dataframe['filepath'].values, labels_onehot))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds

# v5: ยังไม่สร้าง dataset ตรงนี้ เพราะ train/val เปลี่ยนไปทุก fold
# จะเรียก build_dataset() ข้างในฟังก์ชัน เทรน1fold() แทน
test_ds = build_dataset(test_df)   # test set ชุดเดียวใช้ตลอด สร้างไว้เลยได้
print("สร้างฟังก์ชัน pipeline สำเร็จ | test_ds พร้อมแล้ว")


## Cell 10 — mixed precision + ชั้น Augmentation — เหมือน v4.3

**ตอบข้อที่เพื่อนแนะนำเรื่อง "แก้ class imbalance ด้วย data augmentation"**

Augmentation ที่ทาเท่ากันทุกคลาส **ไม่ได้แก้ imbalance** เพราะคลาสใหญ่กับคลาสเล็ก
ถูกเพิ่มความหลากหลายเท่ากัน อัตราส่วน 345:107 ก็ยังเท่าเดิม

สิ่งที่แก้ได้จริงคือ **oversample คลาสเล็กก่อน แล้วค่อย augment** ซึ่ง pipeline นี้ทำอยู่แล้ว
และเพราะ augment อยู่ใน `.map()` ของ tf.data โดย**ไม่มี `.cache()` ต่อท้าย** รูปที่ถูกสุ่มซ้ำ
จะถูกพลิก/หมุน/ซูมใหม่ทุก epoch ไม่ใช่ภาพซ้ำเป๊ะ ๆ นี่คือเหตุผลที่ oversampling ไม่ทำให้ overfit

In [ ]:
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')
print("Compute dtype:", mixed_precision.global_policy().compute_dtype)

augment_layer = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', dtype='float32'),
    tf.keras.layers.RandomRotation(0.08, dtype='float32'),
    tf.keras.layers.RandomZoom(0.1, dtype='float32'),
    tf.keras.layers.RandomTranslation(0.05, 0.05, dtype='float32'),
    tf.keras.layers.RandomContrast(0.1, dtype='float32'),
], name='augment_layer')

print("สร้าง augmentation layer สำเร็จ")


## Cell 11 — MixUp (ใช้เฉพาะ Phase 1) — เหมือน v4.3

**ควรได้:** ไม่มี error (cell นี้แค่นิยามฟังก์ชัน ตัวอย่างภาพจะเห็นใน Cell 12)

In [ ]:
def mixup(images, labels, alpha=0.2):
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.float32)
    batch_size = tf.shape(images)[0]
    gamma1 = tf.random.gamma([], alpha, dtype=tf.float32)
    gamma2 = tf.random.gamma([], alpha, dtype=tf.float32)
    lam = gamma1 / (gamma1 + gamma2)
    idx = tf.random.shuffle(tf.range(batch_size))
    images_mixed = lam * images + (1 - lam) * tf.gather(images, idx)
    labels_mixed = lam * labels + (1 - lam) * tf.gather(labels, idx)
    return images_mixed, labels_mixed

def prepare_phase1(images, labels):
    images = augment_layer(images, training=True)
    images, labels = mixup(images, labels, alpha=0.2)
    return images, labels

def prepare_phase23(images, labels):
    # Phase 2-3 ไม่ใช้ MixUp — ช่วงนี้โมเดลต้องเรียนจุดต่างเล็กๆ ระหว่างสายพันธุ์หน้าคล้ายกัน
    # (Ragdoll / Birman) ถ้ายังผสมภาพอยู่ จุดต่างพวกนั้นจะเบลอหายไปพอดี
    images = augment_layer(images, training=True)
    return images, labels

print("เตรียมฟังก์ชัน MixUp / augmentation สำเร็จ")


## Cell 12 — ฟังก์ชันปรับสมดุลข้อมูล (เรียกใหม่ทุก fold) ⭐ จุดที่ต่างจาก v4.3

**เดิม v4.3** คำนวณ oversample กับ class_weight **ครั้งเดียวก่อนแบ่งข้อมูล**

**v5 เปลี่ยนเป็นฟังก์ชัน** เพื่อเรียกใหม่ทุก fold โดยคำนวณจาก**เฉพาะส่วน train ของ fold นั้น**

**ทำไมต้องเปลี่ยน** ถ้าคำนวณจากข้อมูลทั้งก้อนก่อนแบ่ง แปลว่าตอนเทรน fold ที่ 1 โมเดล
ได้รับอิทธิพลจากสถิติของรูปที่อยู่ใน validation ของ fold นั้นไปแล้ว เรียกว่า **data leakage**
ผลคือคะแนน validation จะดูดีเกินจริง ซึ่งทำให้ K-Fold เสียความหมายไปเลย

**ควรได้:** ไม่มี error (นิยามฟังก์ชันอย่างเดียว)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def ปรับสมดุล(train_fold_df):
    """รับ train ของ fold หนึ่ง -> คืน (df ที่ oversample แล้ว, dict ของ class weight)

    ห้ามส่ง validation หรือ test เข้ามาเด็ดขาด
    """
    # --- 1) Oversampling แบบมีเพดาน ---
    # ไม่ดันให้เท่ากันเป๊ะ เพราะคลาสที่มี ~91 รูป ถ้าดันไป ~293 ต้องซ้ำภาพเดิม 3 เท่า
    # โมเดลจะจำ "รูปนั้น" แทนที่จะจำ "สายพันธุ์นั้น"
    max_count = train_fold_df['label'].value_counts().max()
    target = int(max_count * OVERSAMPLE_CAP_RATIO)

    ชิ้นส่วน = []
    for label, group in train_fold_df.groupby('label'):
        if len(group) < target:
            เพิ่ม = group.sample(target - len(group), replace=True, random_state=SEED)
            ชิ้นส่วน.append(pd.concat([group, เพิ่ม]))
        else:
            ชิ้นส่วน.append(group)

    df_os = pd.concat(ชิ้นส่วน).sample(frac=1, random_state=SEED).reset_index(drop=True)

    # --- 2) Class weight คำนวณ "หลัง" oversample ---
    y_idx = df_os['label'].map(label_to_idx).values
    w = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=y_idx)
    weight_dict = {i: float(v) for i, v in enumerate(w)}

    return df_os, weight_dict

print("นิยามฟังก์ชัน ปรับสมดุล() สำเร็จ — จะถูกเรียกใหม่ทุก fold")


## Cell 13 — ฟังก์ชันสร้างโมเดลใหม่ (เรียกใหม่ทุก fold) ⭐

**ทำไมต้องสร้างใหม่ทุก fold** ถ้าใช้โมเดลตัวเดิมเทรนต่อ fold ที่ 2 จะเริ่มจากโมเดลที่
เคยเห็นข้อมูล validation ของตัวเองมาแล้วตอนเป็น train ของ fold ที่ 1 — ค่าที่ได้จะไร้ความหมาย

**ควรได้:** ไม่มี error

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

def สร้างโมเดล():
    base = EfficientNetB0(
        include_top=False, weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False   # Phase 1 ล็อกทั้งก้อน

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    # ชั้นสุดท้ายบังคับ float32 — softmax ที่ float16 จะเสียความละเอียดเชิงตัวเลข
    outputs = layers.Dense(len(CLASS_NAMES), activation='softmax', dtype='float32')(x)

    return models.Model(inputs, outputs), base


def ล็อกBatchNorm(base):
    """ล็อกชั้น BatchNormalization ทุกชั้นเสมอ — จุดที่ทำให้ v4.2 พัง

    BN เก็บค่าสถิติ (moving mean/variance) ที่เรียนมาจาก ImageNet ล้านรูป
    พอตั้ง trainable=True มันจะเลิกใช้ค่านั้น หันไปใช้ค่าเฉลี่ยของ batch ปัจจุบันแทน
    batch ละ 32 รูปน้อยเกินกว่าจะได้ค่าที่นิ่ง ค่าจึงแกว่งทุก step จนทำลาย feature ทั้งชุด
    (v4.2 เห็นชัด: val acc 0.92 -> 0.65 ทันทีที่เข้า Phase 3)
    """
    n = 0
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
            n += 1
    return n

_m, _b = สร้างโมเดล()
print(f"โมเดลตัวอย่าง: {len(_b.layers)} ชั้นใน backbone | BatchNorm {ล็อกBatchNorm(_b)} ชั้น")
del _m, _b


## Cell 14 — Loss + Callback (นิยามครั้งเดียว ใช้ทุก fold)

**ควรได้:** ไม่มี error

`MacroF1Callback` คำนวณ Macro-F1 ท้ายทุก epoch แล้วยัดเข้า `logs` เพื่อให้
`EarlyStopping` กับ `ModelCheckpoint` เอาไปใช้เป็นเกณฑ์ตัดสินได้ — Keras ไม่มี metric นี้ให้ในตัว

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

class MacroF1Callback(tf.keras.callbacks.Callback):
    def __init__(self, val_data):
        super().__init__()
        self.val_data = val_data
        self.ประวัติ = []

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []
        for imgs, labels in self.val_data:
            preds = self.model.predict(imgs, verbose=0)
            y_true.extend(np.argmax(labels.numpy(), axis=1))
            y_pred.extend(np.argmax(preds, axis=1))
        f1 = f1_score(y_true, y_pred, average='macro')
        logs['val_macro_f1'] = f1
        self.ประวัติ.append(float(f1))
        print(f"  -> val_macro_f1: {f1:.4f}")

focal_loss = tf.keras.losses.CategoricalFocalCrossentropy(gamma=1.5, label_smoothing=0.05)
print("เตรียม loss / callback สำเร็จ")


## Cell 15 — ฟังก์ชันเทรน 1 fold (ครบ 3 Phase) ⭐ หัวใจของโน้ตบุ๊กนี้

ฟังก์ชันนี้ทำสิ่งเดียวกับ Cell 13-17 ของ v4.3 ทั้งหมด แต่ห่อเป็นฟังก์ชันเพื่อเรียกซ้ำได้ 5 รอบ
โดย**เริ่มจากศูนย์ใหม่ทุกครั้ง** (โมเดลใหม่ / สมดุลข้อมูลใหม่ / callback ใหม่)

**ควรได้:** ไม่มี error (นิยามฟังก์ชันอย่างเดียว ยังไม่เทรน)

In [ ]:
import time, json, shutil

def เทรน1fold(fold_i, tr_idx, va_idx, verbose=1):
    t0 = time.time()
    print("=" * 70)
    print(f"FOLD {fold_i}/{N_FOLDS}  เริ่ม")
    print("=" * 70)

    # ---- 1) แบ่งข้อมูลของ fold นี้ + ปรับสมดุลจาก "train ของ fold นี้เท่านั้น" ----
    tr_df = dev_df.iloc[tr_idx].reset_index(drop=True)
    va_df = dev_df.iloc[va_idx].reset_index(drop=True)
    tr_os, class_weight_dict = ปรับสมดุล(tr_df)
    print(f"train {len(tr_df):,} -> หลัง oversample {len(tr_os):,} | val {len(va_df):,}")

    # ---- 2) pipeline ----
    train_raw = build_dataset(tr_os, shuffle=True)
    ds_p1 = train_raw.map(prepare_phase1,  num_parallel_calls=AUTOTUNE)
    ds_p23 = train_raw.map(prepare_phase23, num_parallel_calls=AUTOTUNE)
    val_ds_fold = build_dataset(va_df)

    # ---- 3) โมเดลใหม่เอี่ยม ----
    model, base = สร้างโมเดล()

    # ---- 4) callback ใหม่ทุก fold (ห้ามใช้ตัวเดิมข้าม fold) ----
    f1_cb = MacroF1Callback(val_ds_fold)
    early = tf.keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', mode='max', patience=5,
        restore_best_weights=True, verbose=1)
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=0)
    # ModelCheckpoint ไม่รีเซ็ตค่า best ตอนเรียก fit() ใหม่ จึงคร่อมได้ทั้ง 3 phase
    # (EarlyStopping รีเซ็ตทุกครั้ง จึงกู้ได้แค่ดีที่สุด "ภายใน phase")
    ckpt_path = f'/content/fold{fold_i}_best.weights.h5'
    ckpt = tf.keras.callbacks.ModelCheckpoint(
        ckpt_path, monitor='val_macro_f1', mode='max',
        save_best_only=True, save_weights_only=True, verbose=0)
    cbs = [f1_cb, reduce_lr, ckpt, early]

    # ---- Phase 1: หัวโมเดลอย่างเดียว (backbone ล็อกหมด) + MixUp ----
    print(f"\n--- Fold {fold_i} | Phase 1 (LR 1e-3, MixUp เปิด) ---")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=focal_loss, metrics=['accuracy'])
    h1 = model.fit(ds_p1, validation_data=val_ds_fold, epochs=EPOCHS_PHASE1,
                   class_weight=class_weight_dict, callbacks=cbs, verbose=verbose)
    e1 = len(h1.history['loss'])

    # ---- Phase 2: ปลด 60 ชั้นท้ายของ backbone (ยังล็อก BatchNorm) ----
    print(f"\n--- Fold {fold_i} | Phase 2 (LR 1e-4, ปลด 60 ชั้นท้าย, MixUp ปิด) ---")
    base.trainable = True
    for layer in base.layers[:-60]:
        layer.trainable = False
    nbn = ล็อกBatchNorm(base)
    print(f"ปลด freeze {sum(1 for l in base.layers if l.trainable)}/{len(base.layers)} ชั้น (ล็อก BN {nbn} ชั้น)")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=focal_loss, metrics=['accuracy'])
    h2 = model.fit(ds_p23, validation_data=val_ds_fold, initial_epoch=e1,
                   epochs=EPOCHS_PHASE2, class_weight=class_weight_dict,
                   callbacks=cbs, verbose=verbose)
    e2 = e1 + len(h2.history['loss'])

    # ---- Phase 3: ปลดทั้งหมด (ยังล็อก BatchNorm) LR ต่ำมาก ----
    print(f"\n--- Fold {fold_i} | Phase 3 (LR 1e-6, ปลดทั้งหมด) ---")
    for layer in base.layers:
        layer.trainable = True
    ล็อกBatchNorm(base)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-6), loss=focal_loss, metrics=['accuracy'])
    h3 = model.fit(ds_p23, validation_data=val_ds_fold, initial_epoch=e2,
                   epochs=e2 + EPOCHS_PHASE3_EXTRA, class_weight=class_weight_dict,
                   callbacks=cbs, verbose=verbose)
    e3 = e2 + len(h3.history['loss'])

    # ---- กู้ weight ที่ดีที่สุดของทั้ง 3 phase กลับมา แล้ววัดผลบน val ของ fold นี้ ----
    model.load_weights(ckpt_path)
    y_true, y_pred = [], []
    for imgs, labels in val_ds_fold:
        p = model.predict(imgs, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(p, axis=1))

    ผล = {
        'fold': fold_i,
        'val_accuracy': float(accuracy_score(y_true, y_pred)),
        'val_macro_f1': float(f1_score(y_true, y_pred, average='macro')),
        'n_train': int(len(tr_df)), 'n_train_oversampled': int(len(tr_os)),
        'n_val': int(len(va_df)),
        'epochs_p1': int(e1), 'epochs_p2': int(e2 - e1), 'epochs_p3': int(e3 - e2),
        'ประวัติ_val_macro_f1': f1_cb.ประวัติ,
        'นาทีที่ใช้': round((time.time() - t0) / 60, 1),
    }

    # สำเนา weight ขึ้น Drive — ดิสก์ Colab หายเมื่อ runtime หลุด
    shutil.copy(ckpt_path, f'{SAVE_PATH}/fold{fold_i}_best.weights.h5')

    print(f"\nFOLD {fold_i} เสร็จ | val_acc {ผล['val_accuracy']:.4f} | "
          f"val_macro_f1 {ผล['val_macro_f1']:.4f} | ใช้เวลา {ผล['นาทีที่ใช้']} นาที")

    del model, base
    tf.keras.backend.clear_session()
    return ผล

print("นิยามฟังก์ชัน เทรน1fold() สำเร็จ")


## Cell 16 — รัน K-Fold (รันซ้ำได้ ไม่เทรนซ้ำของเดิม) ⭐

**Cell นี้ใช้เวลานานมาก ประมาณ 1.5 ชั่วโมงต่อ fold**

**ถ้า Colab หลุดกลางทาง ไม่ต้องตกใจ** ผลของ fold ที่เทรนจบแล้วถูกบันทึกลง Drive ทุกครั้ง
กลับมาเปิดใหม่แล้วรัน Cell 1 ถึง 15 ใหม่ แล้วรัน cell นี้อีกที มันจะข้าม fold ที่เสร็จแล้วให้เอง

**อยากรันทีละ fold** ให้แก้บรรทัด `จำกัดรอบนี้ = None` เป็น `จำกัดรอบนี้ = 1`
แล้วรันวันละครั้ง จนครบ 5 fold

**ควรได้:** เห็น log การเทรนไล่ไปทีละ fold และท้ายสุดพิมพ์ว่าเสร็จกี่ fold แล้ว

In [ ]:
RESULTS_PATH = f'{SAVE_PATH}/cv_results.json'

จำกัดรอบนี้ = None   # None = รันจนครบ | ใส่ 1 = รันแค่ 1 fold แล้วหยุด

ผลทั้งหมด = []
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, encoding='utf-8') as f:
        ผลทั้งหมด = json.load(f)
    print(f"เจอผลเดิมใน Drive: เทรนไปแล้ว {len(ผลทั้งหมด)} fold -> {[r['fold'] for r in ผลทั้งหมด]}")

เสร็จแล้ว = {r['fold'] for r in ผลทั้งหมด}
รันไปกี่รอบ = 0

for i, (tr_idx, va_idx) in enumerate(folds):
    fold_i = i + 1
    if fold_i in เสร็จแล้ว:
        print(f"ข้าม Fold {fold_i} (เทรนจบแล้ว)")
        continue
    if จำกัดรอบนี้ is not None and รันไปกี่รอบ >= จำกัดรอบนี้:
        print(f"หยุดตามที่ตั้งไว้ ({จำกัดรอบนี้} fold ต่อรอบ) — รัน cell นี้ใหม่เพื่อไปต่อ")
        break

    ผล = เทรน1fold(fold_i, tr_idx, va_idx)
    ผลทั้งหมด.append(ผล)
    ผลทั้งหมด.sort(key=lambda r: r['fold'])

    # เขียนลง Drive ทันทีหลังจบแต่ละ fold — นี่คือสิ่งที่ทำให้ resume ได้
    with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
        json.dump(ผลทั้งหมด, f, ensure_ascii=False, indent=2)
    print(f"บันทึกผล Fold {fold_i} ลง Drive แล้ว\n")
    รันไปกี่รอบ += 1

print(f"\nสถานะตอนนี้: เทรนเสร็จ {len(ผลทั้งหมด)}/{N_FOLDS} fold")
if len(ผลทั้งหมด) < N_FOLDS:
    print("ยังไม่ครบ — รัน cell นี้ใหม่อีกครั้งเพื่อเทรน fold ที่เหลือ")


## Cell 17 — สรุปผล K-Fold: mean ± std ⭐ ตัวเลขที่ต้องเอาไปใส่เล่ม

**นี่คือคำตอบของคำถามที่เพื่อนถามว่า "รู้ได้ไงว่า accuracy ดีจริง ไม่ใช่ฟลุค"**

Holdout เดิมให้ตัวเลขเดียว บอกไม่ได้ว่าถ้าแบ่งข้อมูลใหม่จะได้เท่าไร
K-Fold ให้ 5 ตัวเลขจากการแบ่ง 5 แบบ **ส่วนเบี่ยงเบนมาตรฐาน (std) คือคำตอบ**

- std น้อย (< 0.01) = ผลเสถียร ไม่ว่าจะแบ่งยังไงก็ได้ประมาณนี้
- std เยอะ (> 0.03) = ผลขึ้นกับว่าบังเอิญแบ่งได้แบบไหน ตัวเลขเดียวจาก holdout เชื่อไม่ได้

**ควรได้:** ตาราง 5 แถว + บรรทัดสรุป mean ± std + กราฟเส้น val_macro_f1 ของทั้ง 5 fold

In [ ]:
import pandas as pd, numpy as np

with open(RESULTS_PATH, encoding='utf-8') as f:
    ผลทั้งหมด = json.load(f)

if len(ผลทั้งหมด) < N_FOLDS:
    print(f"เตือน: มีผลแค่ {len(ผลทั้งหมด)}/{N_FOLDS} fold — ตัวเลขข้างล่างยังไม่สมบูรณ์")

ตาราง = pd.DataFrame([{
    'Fold': r['fold'],
    'Train': r['n_train'], 'Val': r['n_val'],
    'Val Accuracy': round(r['val_accuracy'], 4),
    'Val Macro-F1': round(r['val_macro_f1'], 4),
    'Epochs (P1/P2/P3)': f"{r['epochs_p1']}/{r['epochs_p2']}/{r['epochs_p3']}",
    'นาที': r['นาทีที่ใช้'],
} for r in ผลทั้งหมด])
display(ตาราง)

acc = np.array([r['val_accuracy'] for r in ผลทั้งหมด])
f1  = np.array([r['val_macro_f1'] for r in ผลทั้งหมด])

print("=" * 62)
print(f"Cross-Validation ({len(ผลทั้งหมด)}-Fold)")
print(f"  Accuracy  : {acc.mean():.4f} +/- {acc.std():.4f}   (ต่ำสุด {acc.min():.4f} / สูงสุด {acc.max():.4f})")
print(f"  Macro-F1  : {f1.mean():.4f} +/- {f1.std():.4f}   (ต่ำสุด {f1.min():.4f} / สูงสุด {f1.max():.4f})")
print("=" * 62)
print(f"\nเขียนในเล่มได้ว่า: ความแม่นยำ {acc.mean()*100:.2f}% +/- {acc.std()*100:.2f}%")

ตาราง.to_csv(f'{SAVE_PATH}/cv_summary.csv', index=False, encoding='utf-8-sig')

# กราฟ: val_macro_f1 ต่อ epoch ของทุก fold ซ้อนกัน
plt.figure(figsize=(11, 5))
for r in ผลทั้งหมด:
    plt.plot(range(1, len(r['ประวัติ_val_macro_f1']) + 1), r['ประวัติ_val_macro_f1'],
             marker='o', ms=3, label=f"Fold {r['fold']}")
plt.axvline(EPOCHS_PHASE1, color='gray', ls='--', alpha=.6, label='เริ่ม Phase 2')
plt.xlabel('รอบการเรียนรู้ (Epoch)'); plt.ylabel('Macro-F1 ของชุดตรวจสอบ')
plt.title(f'Macro-F1 ของชุดตรวจสอบ แยกตาม Fold ({len(ผลทั้งหมด)}-Fold Cross-Validation)')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f'{SAVE_PATH}/cv_macro_f1_per_fold.png', dpi=150, bbox_inches='tight')
plt.show()


## Cell 18 — ทำไม Validation 15% แบบเดิมถึงบอกไม่ได้ว่า "แม่นแล้ว" ⭐

**ตอบข้อที่เพื่อนถามโดยตรง** cell นี้คำนวณช่วงความเชื่อมั่น 95% (Wilson score interval)
ให้เห็นเป็นตัวเลขว่า ชุดประเมินขนาดเล็กทำให้ตัวเลขแกว่งได้แค่ไหน

**ควรได้:** ตารางเทียบ — ค่ารวมช่วงแคบ (เชื่อได้) แต่ค่ารายสายพันธุ์หายากช่วงกว้างมาก (เชื่อไม่ได้)

In [ ]:
from math import sqrt

def wilson(ถูก, ทั้งหมด, z=1.96):
    """ช่วงความเชื่อมั่นของสัดส่วน — ใช้ Wilson เพราะแม่นกว่าสูตรปกติเมื่อ n น้อย"""
    if ทั้งหมด == 0: return (0.0, 0.0)
    p = ถูก / ทั้งหมด
    d = 1 + z*z/ทั้งหมด
    กลาง = (p + z*z/(2*ทั้งหมด)) / d
    ครึ่ง = z*sqrt(p*(1-p)/ทั้งหมด + z*z/(4*ทั้งหมด*ทั้งหมด)) / d
    return ((กลาง-ครึ่ง)*100, (กลาง+ครึ่ง)*100)

แถว = []
# 1) ภาพรวมบน test set
n_test = len(test_df)
for ชื่อ, n, p in [
    (f'ภาพรวม (holdout เดิม, n={n_test})', n_test, 0.934),
]:
    lo, hi = wilson(round(p*n), n)
    แถว.append({'สิ่งที่วัด': ชื่อ, 'n': n, 'ค่าที่วัดได้': f'{p*100:.1f}%',
                'ช่วงความเชื่อมั่น 95%': f'{lo:.1f}% - {hi:.1f}%', 'กว้าง (จุด)': round(hi-lo, 1)})

# 2) รายสายพันธุ์ — ใช้จำนวนจริงของ 3 สายพันธุ์ที่ข้อมูลน้อยสุด
น้อยสุด = df['label'].value_counts().tail(3)
for ชื่อ, ทั้งหมด in น้อยสุด.items():
    n_holdout = round(ทั้งหมด * TEST_RATIO)          # แบบเดิม: ประเมินจาก test 15%
    n_cv      = round(ทั้งหมด * (1 - TEST_RATIO))    # แบบ K-Fold: ทุกรูปใน dev ได้เป็น val
    for ป้าย, n in [(f'{ชื่อ} (holdout 15%)', n_holdout), (f'{ชื่อ} (K-Fold)', n_cv)]:
        lo, hi = wilson(round(0.85*n), n)
        แถว.append({'สิ่งที่วัด': ป้าย, 'n': n, 'ค่าที่วัดได้': '85.0%',
                    'ช่วงความเชื่อมั่น 95%': f'{lo:.1f}% - {hi:.1f}%', 'กว้าง (จุด)': round(hi-lo, 1)})

display(pd.DataFrame(แถว))
print("""
อ่านตารางนี้ยังไง
  - ค่าภาพรวมช่วงแคบประมาณ +/- 1.5 จุด  -> ตัวเลข 93.4% เชื่อถือได้
  - ค่ารายสายพันธุ์หายากแบบ holdout ช่วงกว้างมาก เพราะเหลือประเมินแค่ 15-16 รูป
    ทายผิดเพิ่ม 1 รูป ค่าตกทันที 6 จุด -> ตัวเลข 0.88 กับ 0.82 แทบแยกกันไม่ออกทางสถิติ
  - พอใช้ K-Fold ทุกรูปใน dev set ได้เป็น validation หนึ่งครั้ง n โตขึ้นประมาณ 5 เท่า
    ช่วงความเชื่อมั่นจึงแคบลงชัดเจน นี่คือเหตุผลหลักที่ต้องทำ K-Fold
""")


## Cell 19 — เลือกโมเดลสุดท้าย แล้ววัดผลบน Test set ที่ล็อกไว้ ⭐

**ตรงนี้คือครั้งแรกและครั้งเดียวที่โมเดลได้เจอ Test set**

วิธีเลือกโมเดลสุดท้ายมี 2 ทาง โน้ตบุ๊กนี้ใช้ทาง A (ไม่ต้องเทรนเพิ่ม)

- **ทาง A — เลือก fold ที่ Macro-F1 สูงสุด** ใช้ weight ของ fold นั้นเป็นโมเดล production
  ข้อดี: ไม่ต้องเทรนเพิ่มอีก 1.5 ชม. | ข้อเสีย: ใช้ข้อมูลเทรนแค่ 80% ของ dev set
- **ทาง B — เทรนใหม่บน dev set ทั้งก้อน** โดยใช้จำนวน epoch เฉลี่ยจาก CV
  ข้อดี: ได้ข้อมูลเทรนครบ 100% มักแม่นกว่านิดหน่อย | ข้อเสีย: ต้องเทรนเพิ่มอีกรอบ
  (ถ้าอยากทำทาง B มีโค้ดให้ใน Cell 20)

**ควรได้:** Test Accuracy กับ Macro-F1 ซึ่งควรใกล้เคียงค่า mean จาก CV
ถ้าต่ำกว่า CV มากผิดปกติ แปลว่ามีอะไรรั่วระหว่างทาง

In [ ]:
ดีที่สุด = max(ผลทั้งหมด, key=lambda r: r['val_macro_f1'])
print(f"เลือก Fold {ดีที่สุด['fold']} (val_macro_f1 = {ดีที่สุด['val_macro_f1']:.4f})")

final_model, final_base = สร้างโมเดล()
final_base.trainable = True
ล็อกBatchNorm(final_base)
final_model.load_weights(f"{SAVE_PATH}/fold{ดีที่สุด['fold']}_best.weights.h5")
print("โหลด weight สำเร็จ")

y_true, y_pred, y_proba = [], [], []
for imgs, labels in test_ds:
    p = final_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(p, axis=1))
    y_proba.extend(p)

y_true = np.array(y_true); y_pred = np.array(y_pred); y_proba = np.array(y_proba)

test_acc = accuracy_score(y_true, y_pred)
test_f1  = f1_score(y_true, y_pred, average='macro')
top3 = np.mean([y_true[i] in np.argsort(y_proba[i])[-3:] for i in range(len(y_true))])

lo, hi = wilson(int((y_true == y_pred).sum()), len(y_true))
print("=" * 62)
print(f"ผลบน TEST SET ({len(y_true):,} รูป ที่ไม่เคยถูกใช้เลยตลอดงาน)")
print(f"  Top-1 Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)   95% CI {lo:.1f}% - {hi:.1f}%")
print(f"  Top-3 Accuracy : {top3:.4f}  ({top3*100:.2f}%)")
print(f"  Macro-F1       : {test_f1:.4f}")
print("-" * 62)
print(f"  เทียบกับ CV    : acc {acc.mean():.4f} +/- {acc.std():.4f} | f1 {f1.mean():.4f} +/- {f1.std():.4f}")
print("=" * 62)


## Cell 20 — (ทางเลือก) เทรนใหม่บน dev set ทั้งก้อน — ข้ามได้ถ้าไม่มีเวลา

ใช้เมื่ออยากได้โมเดลที่เห็นข้อมูลครบ 100% ของ dev set ใช้เวลาเพิ่มประมาณ 1.5 ชั่วโมง
จำนวน epoch ใช้ค่ามัธยฐานจาก 5 fold เพราะไม่มี validation ให้ EarlyStopping ดูแล้ว

**ถ้าไม่รัน cell นี้ ให้ข้ามไป Cell 21 ได้เลย** โมเดลจาก Cell 19 ใช้งานได้ปกติ

In [ ]:
รันcellนี้ = False   # เปลี่ยนเป็น True ถ้าต้องการเทรนทับ

if รันcellนี้:
    med_p1 = int(np.median([r['epochs_p1'] for r in ผลทั้งหมด]))
    med_p2 = int(np.median([r['epochs_p2'] for r in ผลทั้งหมด]))
    med_p3 = int(np.median([r['epochs_p3'] for r in ผลทั้งหมด]))
    print(f"ใช้ epoch มัธยฐานจาก CV -> P1={med_p1} P2={med_p2} P3={med_p3}")

    dev_os, cw = ปรับสมดุล(dev_df)
    raw = build_dataset(dev_os, shuffle=True)
    d1, d23 = raw.map(prepare_phase1, num_parallel_calls=AUTOTUNE), raw.map(prepare_phase23, num_parallel_calls=AUTOTUNE)

    final_model, final_base = สร้างโมเดล()
    final_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=focal_loss, metrics=['accuracy'])
    final_model.fit(d1, epochs=med_p1, class_weight=cw)

    final_base.trainable = True
    for l in final_base.layers[:-60]: l.trainable = False
    ล็อกBatchNorm(final_base)
    final_model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=focal_loss, metrics=['accuracy'])
    final_model.fit(d23, initial_epoch=med_p1, epochs=med_p1+med_p2, class_weight=cw)

    for l in final_base.layers: l.trainable = True
    ล็อกBatchNorm(final_base)
    final_model.compile(optimizer=tf.keras.optimizers.Adam(1e-6), loss=focal_loss, metrics=['accuracy'])
    final_model.fit(d23, initial_epoch=med_p1+med_p2, epochs=med_p1+med_p2+med_p3, class_weight=cw)

    final_model.save_weights(f'{SAVE_PATH}/final_full_dev.weights.h5')
    print("เทรนบน dev set ทั้งก้อนเสร็จ -> ให้กลับไปรัน Cell 19 ใหม่เพื่อวัดผลบน test")
else:
    print("ข้าม cell นี้ (ใช้โมเดลจาก fold ที่ดีที่สุดตาม Cell 19)")


## Cell 21 — ตาราง Precision / Recall / F1 ทุกสายพันธุ์ (บน Test set)

**ควรได้:** ตารางความร้อน 31 แถว เรียงจาก F1 น้อยไปมาก

In [ ]:
from sklearn.metrics import classification_report
import seaborn as sns

rep = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
rep_df = pd.DataFrame(rep).transpose()
คลาส = rep_df.loc[CLASS_NAMES].sort_values('f1-score')
สรุปท้าย = rep_df.loc[['accuracy', 'macro avg', 'weighted avg']]
ตารางรวม = pd.concat([คลาส[['precision','recall','f1-score']],
                      สรุปท้าย[['precision','recall','f1-score']]])

plt.figure(figsize=(9, 12))
sns.heatmap(ตารางรวม, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, cbar=False,
            linewidths=.4, linecolor='white')
plt.title('ตารางสรุปผลการทำนาย (Precision / Recall / F1-Score) บน Test set', fontsize=14)
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}/classification_report.png', dpi=150, bbox_inches='tight')
plt.show()

ตารางรวม.to_csv(f'{SAVE_PATH}/classification_report.csv', encoding='utf-8-sig')


## Cell 22 — Confusion Matrix (สัดส่วนต่อแถว) + คู่ที่สับสนมากที่สุด

**ควรได้:** ตาราง 31x31 และรายการ 10 คู่ที่สับสนกันมากที่สุด

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
cm_df = pd.DataFrame(cm_norm, index=CLASS_NAMES, columns=CLASS_NAMES)

plt.figure(figsize=(22, 20))
sns.heatmap(cm_df, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, cbar=True,
            linewidths=.3, linecolor='white', annot_kws={'size': 5})
plt.title('เมทริกซ์สับสน (Confusion Matrix) — สัดส่วนต่อแถว', fontsize=16)
plt.xlabel('ค่าที่ทำนายได้'); plt.ylabel('ค่าที่ใช่')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}/confusion_matrix_normalized.png', dpi=150)
plt.show()

สับสน = []
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        if i != j and cm[i, j] > 0:
            สับสน.append({'ของจริง': CLASS_NAMES[i], 'ทายผิดเป็น': CLASS_NAMES[j],
                          'จำนวนครั้ง': int(cm[i, j]), '% ของจริงกลุ่มนี้': round(cm_norm[i, j]*100, 1)})
สับสน_df = pd.DataFrame(สับสน).sort_values('จำนวนครั้ง', ascending=False)
สับสน_df.to_csv(f'{SAVE_PATH}/top_confusions.csv', index=False, encoding='utf-8-sig')
print("10 คู่สายพันธุ์ที่สับสนกันมากที่สุด:")
display(สับสน_df.head(10))


## Cell 23 — บันทึกโมเดล + แปลงเป็น ONNX + metadata.json

**ควรได้:** ไฟล์ `final_model.onnx` กับ `metadata.json` ใน Drive พร้อมเอาไปวางใน `ai_server/model/`

In [ ]:
!pip install -q tf2onnx onnx

import tf2onnx, json

final_model.save(f'{SAVE_PATH}/model.keras')

spec = (tf.TensorSpec((None, IMG_SIZE, IMG_SIZE, 3), tf.float32, name='input'),)
onnx_model, _ = tf2onnx.convert.from_keras(final_model, input_signature=spec, opset=13)
with open(f'{SAVE_PATH}/final_model.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())

# ---- breed_info ภาษาไทย (จัด pattern แล้ว: นิสัย 3 คำ / ขนาด 3 ระดับ / ชื่อไทยไม่มีวงเล็บ) ----
# ฝังไว้ในโน้ตบุ๊กเลย เพราะ src/data/breeds.js ฝั่งเว็บ import metadata.json ไปใช้โดยตรง
# ถ้าไฟล์ที่สร้างจากที่นี่ไม่มี breed_info เว็บจะ build ไม่ผ่านทันที
BREED_INFO = {
    "american_bulldog": {
        "ชื่อไทย": "อเมริกัน บูลด็อก",
        "ประเภท": "สุนัข",
        "ขนาด": "ใหญ่",
        "นิสัย": "มั่นใจ / ตื่นตัว / รักครอบครัว"
    },
    "american_pit_bull_terrier": {
        "ชื่อไทย": "อเมริกัน พิตบูล เทอร์เรีย",
        "ประเภท": "สุนัข",
        "ขนาด": "ใหญ่",
        "นิสัย": "มั่นใจ / พลังงานสูง / ซื่อสัตย์",
        "หมายเหตุ": "ต้องการการฝึกและวินัยสม่ำเสมอ"
    },
    "beagle": {
        "ชื่อไทย": "บีเกิล",
        "ประเภท": "สุนัข",
        "ขนาด": "กลาง",
        "นิสัย": "ขี้เล่น / เป็นมิตร / พลังงานสูง"
    },
    "bengal": {
        "ชื่อไทย": "เบงกอล",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "มั่นใจ / พลังงานสูง / ขี้เล่น",
        "หมายเหตุ": "ลายจุดหรือลายหินอ่อนคล้ายเสือดาว"
    },
    "birman": {
        "ชื่อไทย": "เบอร์แมน",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "อ่อนโยน / เงียบสงบ / เป็นมิตร"
    },
    "british_shorthair": {
        "ชื่อไทย": "บริติช ช็อตแฮร์",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "สงบ / สุขุม / รักอิสระ"
    },
    "calico": {
        "ชื่อไทย": "แมวสามสี",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "เป็นมิตร / ขี้เล่น / ปรับตัวเก่ง",
        "ชื่ออื่น": "คาลิโก",
        "หมายเหตุ": "เป็นลายขนสามสี ไม่ใช่สายพันธุ์เฉพาะ"
    },
    "chihuahua": {
        "ชื่อไทย": "ชิวาวา",
        "ประเภท": "สุนัข",
        "ขนาด": "เล็ก",
        "นิสัย": "ตื่นตัว / คล่องแคล่ว / ผูกพันเจ้าของ"
    },
    "domestic_shorthair": {
        "ชื่อไทย": "แมวบ้านขนสั้น",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "ปรับตัวเก่ง / เป็นมิตร / เลี้ยงง่าย",
        "หมายเหตุ": "แมวผสมทั่วไป พบมากตามท้องถนน"
    },
    "english_cocker_spaniel": {
        "ชื่อไทย": "อิงลิช ค็อกเกอร์ สเปเนียล",
        "ประเภท": "สุนัข",
        "ขนาด": "กลาง",
        "นิสัย": "ร่าเริง / อ่อนโยน / เชื่อฟัง"
    },
    "golden_retriever": {
        "ชื่อไทย": "โกลเด้น รีทรีฟเวอร์",
        "ประเภท": "สุนัข",
        "ขนาด": "ใหญ่",
        "นิสัย": "เป็นมิตร / อ่อนโยน / ฉลาด"
    },
    "havanese": {
        "ชื่อไทย": "ฮาวาเนส",
        "ประเภท": "สุนัข",
        "ขนาด": "เล็ก",
        "นิสัย": "เป็นมิตร / ขี้เล่น / ติดเจ้าของ"
    },
    "japanese_chin": {
        "ชื่อไทย": "เจแปนนิส ชิน",
        "ประเภท": "สุนัข",
        "ขนาด": "เล็ก",
        "นิสัย": "รักสงบ / อ่อนโยน / เงียบ"
    },
    "khao_manee": {
        "ชื่อไทย": "ขาวมณี",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "ฉลาด / ขี้เล่น / เป็นมิตร",
        "หมายเหตุ": "แมวมงคลไทย"
    },
    "konja": {
        "ชื่อไทย": "โกญจา",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "ฉลาด / ขี้อ้อน / ผูกพันเจ้าของ",
        "หมายเหตุ": "แมวมงคลไทย หายาก"
    },
    "korat": {
        "ชื่อไทย": "โคราช",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "ขี้อ้อน / ผูกพันเจ้าของ / ฉลาด",
        "ชื่ออื่น": "สีสวาด",
        "หมายเหตุ": "แมวมงคลไทย เชื่อว่านำโชค"
    },
    "labrador_retriever": {
        "ชื่อไทย": "ลาบราดอร์ รีทรีฟเวอร์",
        "ประเภท": "สุนัข",
        "ขนาด": "ใหญ่",
        "นิสัย": "เป็นมิตร / ร่าเริง / ฉลาด",
        "หมายเหตุ": "ชอบเล่นน้ำ"
    },
    "maine_coon": {
        "ชื่อไทย": "เมนคูน",
        "ประเภท": "แมว",
        "ขนาด": "ใหญ่",
        "นิสัย": "เป็นมิตร / ขี้เล่น / สุขุม",
        "หมายเหตุ": "แมวขนาดใหญ่ที่สุดในบรรดาแมวบ้าน"
    },
    "persian": {
        "ชื่อไทย": "เปอร์เซีย",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "สงบ / อ่อนโยน / รักความเงียบสงบ"
    },
    "pomeranian": {
        "ชื่อไทย": "ปอมเมอเรเนียน",
        "ประเภท": "สุนัข",
        "ขนาด": "เล็ก",
        "นิสัย": "ร่าเริง / ตื่นตัว / ฉลาด"
    },
    "pug": {
        "ชื่อไทย": "ปั๊ก",
        "ประเภท": "สุนัข",
        "ขนาด": "เล็ก",
        "นิสัย": "ตลก / ร่าเริง / ปรับตัวง่าย"
    },
    "ragdoll": {
        "ชื่อไทย": "แร็กดอลล์",
        "ประเภท": "แมว",
        "ขนาด": "ใหญ่",
        "นิสัย": "เชื่องมาก / อ่อนโยน / ชอบให้อุ้ม"
    },
    "shiba_inu": {
        "ชื่อไทย": "ชิบะอินุ",
        "ประเภท": "สุนัข",
        "ขนาด": "กลาง",
        "นิสัย": "ฉลาด / รักสะอาด / เป็นตัวของตัวเอง"
    },
    "siamese": {
        "ชื่อไทย": "วิเชียรมาศ",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "ฉลาด / ช่างพูด / ผูกพันเจ้าของ",
        "หมายเหตุ": "แมวมงคลไทย"
    },
    "siberian_husky": {
        "ชื่อไทย": "ไซบีเรียน ฮัสกี้",
        "ประเภท": "สุนัข",
        "ขนาด": "ใหญ่",
        "นิสัย": "ร่าเริง / พลังงานสูงมาก / เป็นมิตร",
        "หมายเหตุ": "ต้องการพื้นที่วิ่งและอากาศไม่ร้อนจัด"
    },
    "suphalak": {
        "ชื่อไทย": "ศุภลักษณ์",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "สงบ / ผูกพันเจ้าของ / อ่อนโยน",
        "หมายเหตุ": "แมวมงคลไทย หายาก"
    },
    "thai_bangkaew": {
        "ชื่อไทย": "ไทยบางแก้ว",
        "ประเภท": "สุนัข",
        "ขนาด": "กลาง",
        "นิสัย": "ฉลาด / ซื่อสัตย์ / ปกป้องเจ้าของ",
        "หมายเหตุ": "ระแวงคนแปลกหน้า ควรฝึกเข้าสังคมตั้งแต่เล็ก"
    },
    "thai_mixed_breed_dog": {
        "ชื่อไทย": "หมาไทยพันธุ์ผสม",
        "ประเภท": "สุนัข",
        "ขนาด": "กลาง",
        "นิสัย": "ปรับตัวเก่ง / ซื่อสัตย์ / แข็งแรงทนทาน",
        "หมายเหตุ": "สุนัขผสมทั่วไป พบมากตามท้องถนน"
    },
    "thai_ridgeback": {
        "ชื่อไทย": "ไทยหลังอาน",
        "ประเภท": "สุนัข",
        "ขนาด": "ใหญ่",
        "นิสัย": "ฉลาด / ปราดเปรียว / ซื่อสัตย์",
        "หมายเหตุ": "ระแวงคนแปลกหน้า ควรฝึกเข้าสังคมตั้งแต่เล็ก"
    },
    "tortoiseshell": {
        "ชื่อไทย": "แมวสีเปรอะ",
        "ประเภท": "แมว",
        "ขนาด": "กลาง",
        "นิสัย": "มั่นใจ / มีเอกลักษณ์ / อารมณ์ชัดเจน",
        "ชื่ออื่น": "ลายกระดองเต่า",
        "หมายเหตุ": "เป็นลายขน ไม่ใช่สายพันธุ์เฉพาะ"
    },
    "yorkshire_terrier": {
        "ชื่อไทย": "ยอร์กเชียร์ เทอร์เรีย",
        "ประเภท": "สุนัข",
        "ขนาด": "เล็ก",
        "นิสัย": "กล้าหาญ / มั่นใจ / ขี้อ้อน"
    }
}

ขาด = [c for c in CLASS_NAMES if c not in BREED_INFO]
assert not ขาด, f"BREED_INFO ไม่มีข้อมูลของ: {ขาด}"

metadata = {
    'version': RUN_NAME,
    'classes': CLASS_NAMES,
    'img_size': IMG_SIZE,
    'test_accuracy': float(test_acc),
    'test_macro_f1': float(test_f1),
    'test_top3_accuracy': float(top3),
    'cv_folds': int(len(ผลทั้งหมด)),
    'cv_accuracy_mean': float(acc.mean()), 'cv_accuracy_std': float(acc.std()),
    'cv_macro_f1_mean': float(f1.mean()), 'cv_macro_f1_std': float(f1.std()),
    'breed_info': {c: BREED_INFO[c] for c in CLASS_NAMES},   # เรียงตาม classes
}
with open(f'{SAVE_PATH}/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("บันทึกครบแล้วที่", SAVE_PATH)
print(f"  Test accuracy {test_acc*100:.2f}% | CV {acc.mean()*100:.2f}% +/- {acc.std()*100:.2f}%")
print(f"  breed_info ครบ {len(metadata['breed_info'])}/{len(CLASS_NAMES)} สายพันธุ์")
print("")
print("เอา final_model.onnx กับ metadata.json ไปวางทับใน ai_server/model/ ได้เลย")
